# Paper Experiment Template (Colab)

## Objective
- Run one WOSAC-focused experiment variant with Colab-resumable execution.
- Keep notebook orchestration-only and move logic to `src/` modules.
- Persist all outputs/checkpoints to Drive-backed storage.


In [ ]:
# Step 1: Repo sync + runtime bootstrap (idempotent)
import os
import subprocess
import sys

REPO_URL = 'https://github.com/achyutmorang/wosac-sim-agents-experiments.git'
REPO_DIR = '/content/wosac-sim-agents-experiments'

if os.path.isdir(REPO_DIR):
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'checkout', 'main'], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only', 'origin', 'main'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', '-b', 'main', REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
for p in (REPO_DIR, os.path.join(REPO_DIR, "src")):
    if p not in sys.path:
        sys.path.insert(0, p)

from src.platform import bootstrap_colab_runtime_with_config, wosac_colab_runtime_config
runtime_cfg = wosac_colab_runtime_config(repo_url=REPO_URL, repo_branch="main", repo_dir=REPO_DIR)
bootstrap = bootstrap_colab_runtime_with_config(runtime_cfg)
print("repo_rev:", bootstrap.repo_sync.repo_rev)

if bootstrap.setup.result.get("restart_required", False):
    raise RuntimeError('Runtime restart required. Rerun this cell after restart.')


In [ ]:
# Step 2: Load config + pack metadata
from src.workflows import bootstrap_experiment_pack, load_experiment_config
EXPERIMENT_SLUG = 'your-experiment-slug'
bundle = bootstrap_experiment_pack(slug=EXPERIMENT_SLUG, repo_root='.')
cfg = load_experiment_config(slug=EXPERIMENT_SLUG, repo_root='.')
print('Experiment:', EXPERIMENT_SLUG)
print('Config run block:', cfg.get('run', {}))
display(bundle.summary_table)


In [ ]:
# Step 3: Fast-fail checks before full run
RUN = cfg.get('run', {})
assert RUN.get('persist_root'), 'persist_root is required'
assert int(RUN.get('n_shards', 1)) >= 1
print('Fast-fail checks passed.')
